# How can we have lazy class attributes?

In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
from types import MethodType
from time import sleep
from abc import ABC, ABCMeta, abstractmethod
from functools import wraps
import inspect

def compute(obj, s, time=1):
    print(f"Computing {s} of {obj} ...", end="")
    sleep(time)
    print("DONE!")
    return "Phew, that was a lot of work!"

In [3]:
class Property:
    "Emulate PyProperty_Type() in Objects/descrobject.c"

    def __init__(self, fget=None, fset=None, fdel=None, doc=None):
        print(
            "Property.__init__:",
            f"{self=}",
            f"{fget=}",
            f"{fset=}",
            f"{fdel=}",
            f"{doc=}",
            sep="\n\t",
        )
        self.fget = fget
        self.fset = fset
        self.fdel = fdel
        if doc is None and fget is not None:
            doc = fget.__doc__
        self.__doc__ = doc
        self.decorated_obj = None

    def __call__(self, *args, **kwargs):
        return self.fget.__call__(*args, **kwargs)

    def __get__(self, obj, objtype=None):
        print("Property.__get__:", f"{self=}", f"{obj=}", f"{objtype=}", sep="\n\t")
        if self.fget is None:
            raise AttributeError("unreadable attribute")
        if obj is None:
            result = self
        else:
            self.decorated_obj = obj
            result = self.fget(obj)
        print(f">>> Property.__get__ ", f"{(obj is None)=}", f"{result=}", sep="\n\t")
        return result

    def __set__(self, obj, value):
        if self.fset is None:
            raise AttributeError("can't set attribute")
        self.fset(obj, value)

    def __delete__(self, obj):
        if self.fdel is None:
            raise AttributeError("can't delete attribute")
        self.fdel(obj)

    def getter(self, fget):
        return type(self)(fget, self.fset, self.fdel, self.__doc__)

    def setter(self, fset):
        return type(self)(self.fget, fset, self.fdel, self.__doc__)

    def deleter(self, fdel):
        return type(self)(self.fget, self.fset, fdel, self.__doc__)

In [4]:
class ClassMethodX(property):
    "Emulate PyClassMethod_Type() in Objects/funcobject.c"

    def __init__(self, *args, **kwargs):
        print(
            f"ClassMethod.__init__:", f"{self=}", f"{args=}", f"{kwargs=}", sep="\n\t"
        )
        super().__init__()

    def __call__(self, *args, **kwargs):
        return self.fget.__call__(*args, **kwargs)

    def __get__(self, obj, cls=None):
        print(f"ClassMethod.__get__:", f"{self=}", f"{obj=}", f"{cls=}", sep="\n\t")
        if cls is None:
            cls = type(obj)
        if hasattr(type(self.fget), "__get__"):
            # result = self.f.__get__(cls)
            result = MethodType(self.fget.__get__, cls).__call__()
        else:
            result = MethodType(self.fget, cls)
        print(
            f">>> ClassMethod.__get__:",
            f"{hasattr(type(self.f), '__get__')=}",
            f"{result=}",
            sep="\n\t",
        )
        return result

In [5]:
class ClassMethod:
    "Emulate PyClassMethod_Type() in Objects/funcobject.c"

    def __init__(self, f):
        print(f"ClassMethod.__init__:", f"{self=}", f"{f=}", sep="\n\t")
        self.f = f

    def __call__(self, *args, **kwargs):
        return self.f.__call__(*args, **kwargs)

    def __get__(self, obj, cls=None):
        print(f"ClassMethod.__get__:", f"{self=}", f"{obj=}", f"{cls=}", sep="\n\t")
        if cls is None:
            cls = type(obj)
        if hasattr(type(self.f), "__get__"):
            # result = self.f.__get__(cls)
            result = MethodType(self.f.__get__, cls).__call__()
        else:
            result = MethodType(self.f, cls)
        print(
            f">>> ClassMethod.__get__:",
            f"{hasattr(type(self.f), '__get__')=}",
            f"{result=}",
            sep="\n\t",
        )
        return result

In [6]:
class ClassPropertyMethod:
    "Emulate PyClassMethod_Type() in Objects/funcobject.c"

    def __init__(self, fget=None, fset=None, fdel=None, doc=None):
        print(
            "ClassPropertyMethod.__init__:",
            f"{self=}",
            f"{fget=}",
            f"{fset=}",
            f"{fdel=}",
            f"{doc=}",
            sep="\n\t",
        )
        self.fget = fget
        self.fset = fset
        self.fdel = fdel
        if doc is None and fget is not None:
            doc = fget.__doc__
        self.__doc__ = doc

    def __get__(self, obj, cls=None):
        print(
            f"ClassPropertyMethod.__get__:",
            f"{self=}",
            f"{obj=}",
            f"{cls=}",
            sep="\n\t",
        )
        if cls is None:
            cls = type(obj)
        if hasattr(type(self.fget), "__get__"):
            # result = self.f.__get__(cls)
            result = MethodType(self.fget.__get__, cls).__call__()
        else:
            result = MethodType(self.fget, cls)
        print(
            f">>> ClassPropertyMethod.__get__:",
            f"{hasattr(type(self.fget), '__get__')=}",
            f"{result=}",
            sep="\n\t",
        )
        return result

    def __set__(self, obj, value):
        if self.fset.fset is None:
            raise AttributeError("can't set attribute")
        self.fset.fset(obj, value)

    def __delete__(self, obj):
        if self.fdel.fdel is None:
            raise AttributeError("can't delete attribute")
        self.fdel.fdel(obj)

    def getter(self, fget):
        return type(self)(fget, self.fset, self.fdel, self.__doc__)

    def setter(self, fset):
        return type(self)(self.fget, fset, self.fdel, self.__doc__)

    def deleter(self, fdel):
        return type(self)(self.fget, self.fset, fdel, self.__doc__)

In [7]:
class ClassProperty(property):
    def __init__(self, fget=None, fset=None, fdel=None, doc=None):
        print(
            "ClassProperty.__init__:",
            f"{self=}",
            f"{fget=}",
            f"{fset=}",
            f"{fdel=}",
            f"{doc=}",
            sep="\n\t",
        )
        # super().__init__(ClassMethod(fget), fset, fdel, doc)
        # self.fget = fget #if fget is None else property(fget) if fget is not None else fget
        # self.fset = fset #if fset is None else property(fset) if fset is not None else fset
        # self.fdel = fdel #if fdel is None else property(fdel) if fdel is not None else fdel
        # if doc is None and fget is not None:
        # doc = fget.__doc__
        # self.__doc__ = doc

    # def __get__(self, obj, cls=None):
    #     print(f"ClassProperty.__get__:", f"{self=}", f"{obj=}", f"{cls=}", sep="\n\t")
    #     return self.fget(obj)
    # if cls is None:
    #     cls = type(obj)
    # if hasattr(type(self.fget), "__get__"):
    #     # result = self.f.__get__(cls)
    #     result = MethodType(self.fget.__get__, cls).__call__()
    # else:
    #     result = MethodType(self.fget, cls)
    # print(
    #     f">>> ClassProperty.__get__:",
    #     f"{hasattr(type(self.fget), '__get__')=}",
    #     f"{result=}",
    #     sep="\n\t",
    # )
    # return property(result).__get__(obj)

In [8]:
class Testit:
    @classmethod
    @property
    def clsprop(cls):
        """Real Class-Property"""
        return f"My name is {cls.__name__}"

    @property
    @classmethod
    def wrongclsprop(cls):
        """Wrong-order Class-Property"""
        return f"My name is {cls.__name__}"

    @Property
    @ClassMethod
    def mywrongclsprop(cls):
        """Wrong-order Class-Property"""
        return f"My name is {cls.__name__}"

    @ClassMethod
    @Property
    def myclsprop(cls):
        """My Class-Property"""
        return f"My name is {cls.__name__}"

    @ClassPropertyMethod
    @Property
    def myclsprop2(cls):
        """My Class-Property"""
        return f"My name is {cls.__name__}"

    @ClassProperty
    def myclsprop3(cls):
        """My Class-Property"""
        return f"My name is {cls.__name__}"

    @property
    def prop(self):
        return 42


display(Testit.__dict__)
Testit.myclsprop, Testit.myclsprop2, Testit.myclsprop3

ClassMethod.__init__:
	self=<__main__.ClassMethod object at 0x7fe80f721520>
	f=<function Testit.mywrongclsprop at 0x7fe80f6d8310>
Property.__init__:
	self=<__main__.Property object at 0x7fe80f6cba90>
	fget=<__main__.ClassMethod object at 0x7fe80f721520>
	fset=None
	fdel=None
	doc=None
Property.__init__:
	self=<__main__.Property object at 0x7fe80f6cb850>
	fget=<function Testit.myclsprop at 0x7fe80f6d8940>
	fset=None
	fdel=None
	doc=None
ClassMethod.__init__:
	self=<__main__.ClassMethod object at 0x7fe80f6cb3a0>
	f=<__main__.Property object at 0x7fe80f6cb850>
Property.__init__:
	self=<__main__.Property object at 0x7fe80f6cb6d0>
	fget=<function Testit.myclsprop2 at 0x7fe80f6dd040>
	fset=None
	fdel=None
	doc=None
ClassPropertyMethod.__init__:
	self=<__main__.ClassPropertyMethod object at 0x7fe80f6c5250>
	fget=<__main__.Property object at 0x7fe80f6cb6d0>
	fset=None
	fdel=None
	doc=None
ClassProperty.__init__:
	self=<__main__.ClassProperty object at 0x7fe80f6c8b80>
	fget=<function Testit.myc

mappingproxy({'__module__': '__main__',
              'clsprop': <classmethod at 0x7fe80f7215b0>,
              'wrongclsprop': <property at 0x7fe80f6d9a40>,
              'mywrongclsprop': <__main__.Property at 0x7fe80f6cba90>,
              'myclsprop': <__main__.ClassMethod at 0x7fe80f6cb3a0>,
              'myclsprop2': <__main__.ClassPropertyMethod at 0x7fe80f6c5250>,
              'myclsprop3': <__main__.ClassProperty at 0x7fe80f6c8b80>,
              'prop': <property at 0x7fe80f6d4860>,
              '__dict__': <attribute '__dict__' of 'Testit' objects>,
              '__weakref__': <attribute '__weakref__' of 'Testit' objects>,
              '__doc__': None})

ClassMethod.__get__:
	self=<__main__.ClassMethod object at 0x7fe80f6cb3a0>
	obj=None
	cls=<class '__main__.Testit'>
Property.__get__:
	self=<__main__.Property object at 0x7fe80f6cb850>
	obj=<class '__main__.Testit'>
	objtype=None
>>> Property.__get__ 
	(obj is None)=False
	result='My name is Testit'
>>> ClassMethod.__get__:
	hasattr(type(self.f), '__get__')=True
	result='My name is Testit'
ClassPropertyMethod.__get__:
	self=<__main__.ClassPropertyMethod object at 0x7fe80f6c5250>
	obj=None
	cls=<class '__main__.Testit'>
Property.__get__:
	self=<__main__.Property object at 0x7fe80f6cb6d0>
	obj=<class '__main__.Testit'>
	objtype=None
>>> Property.__get__ 
	(obj is None)=False
	result='My name is Testit'
>>> ClassPropertyMethod.__get__:
	hasattr(type(self.fget), '__get__')=True
	result='My name is Testit'


('My name is Testit',
 'My name is Testit',
 <__main__.ClassProperty at 0x7fe80f6c8b80>)

In [9]:
Testit.mywrongclsprop

Property.__get__:
	self=<__main__.Property object at 0x7fe80f6cba90>
	obj=None
	objtype=<class '__main__.Testit'>
>>> Property.__get__ 
	(obj is None)=True
	result=<__main__.Property object at 0x7fe80f6cba90>


In [10]:
help(Testit)

ClassMethod.__get__:
	self=<__main__.ClassMethod object at 0x7fe80f6cb3a0>
	obj=None
	cls=<class '__main__.Testit'>
Property.__get__:
	self=<__main__.Property object at 0x7fe80f6cb850>
	obj=<class '__main__.Testit'>
	objtype=None
>>> Property.__get__ 
	(obj is None)=False
	result='My name is Testit'
>>> ClassMethod.__get__:
	hasattr(type(self.f), '__get__')=True
	result='My name is Testit'
ClassMethod.__get__:
	self=<__main__.ClassMethod object at 0x7fe80f6cb3a0>
	obj=None
	cls=<class '__main__.Testit'>
Property.__get__:
	self=<__main__.Property object at 0x7fe80f6cb850>
	obj=<class '__main__.Testit'>
	objtype=None
>>> Property.__get__ 
	(obj is None)=False
	result='My name is Testit'
>>> ClassMethod.__get__:
	hasattr(type(self.f), '__get__')=True
	result='My name is Testit'
ClassMethod.__get__:
	self=<__main__.ClassMethod object at 0x7fe80f6cb3a0>
	obj=None
	cls=<class '__main__.Testit'>
Property.__get__:
	self=<__main__.Property object at 0x7fe80f6cb850>
	obj=<class '__main__.Testit'

In [11]:
inspect.signature(classmethod.__init__)

<Signature (self, /, *args, **kwargs)>

In [12]:
Testit.__dict__["myclsprop3"].__dir__()

['__module__',
 '__init__',
 '__dict__',
 '__weakref__',
 '__doc__',
 '__getattribute__',
 '__get__',
 '__set__',
 '__delete__',
 '__new__',
 'getter',
 'setter',
 'deleter',
 'fget',
 'fset',
 'fdel',
 '__isabstractmethod__',
 '__repr__',
 '__hash__',
 '__str__',
 '__setattr__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__eq__',
 '__ne__',
 '__gt__',
 '__ge__',
 '__reduce_ex__',
 '__reduce__',
 '__subclasshook__',
 '__init_subclass__',
 '__format__',
 '__sizeof__',
 '__dir__',
 '__class__']

In [13]:
inspect.signature(Testit.myclsprop3.__get__)

<Signature (instance, owner, /)>

In [14]:
Testit.myclsprop, Testit.myclsprop2, Testit.myclsprop3

ClassMethod.__get__:
	self=<__main__.ClassMethod object at 0x7fe80f6cb3a0>
	obj=None
	cls=<class '__main__.Testit'>
Property.__get__:
	self=<__main__.Property object at 0x7fe80f6cb850>
	obj=<class '__main__.Testit'>
	objtype=None
>>> Property.__get__ 
	(obj is None)=False
	result='My name is Testit'
>>> ClassMethod.__get__:
	hasattr(type(self.f), '__get__')=True
	result='My name is Testit'
ClassPropertyMethod.__get__:
	self=<__main__.ClassPropertyMethod object at 0x7fe80f6c5250>
	obj=None
	cls=<class '__main__.Testit'>
Property.__get__:
	self=<__main__.Property object at 0x7fe80f6cb6d0>
	obj=<class '__main__.Testit'>
	objtype=None
>>> Property.__get__ 
	(obj is None)=False
	result='My name is Testit'
>>> ClassPropertyMethod.__get__:
	hasattr(type(self.fget), '__get__')=True
	result='My name is Testit'


('My name is Testit',
 'My name is Testit',
 <__main__.ClassProperty at 0x7fe80f6c8b80>)

In [15]:
Testit.__dict__["myclsprop3"].fget

In [16]:
Testit.__dict__["myclsprop2"].__isabstractmethod__ = False

In [17]:
Testit.myclsprop3

In [18]:
inspect.signature(wraps)

<Signature (wrapped, assigned=('__module__', '__name__', '__qualname__', '__doc__', '__annotations__'), updated=('__dict__',))>

## Progress

In [19]:
class Property:
    "Emulate PyProperty_Type() in Objects/descrobject.c"

    def __init__(self, fget=None, fset=None, fdel=None, doc=None):
        print(
            "Property.__init__:",
            f"{self=}",
            f"{fget=}",
            f"{fset=}",
            f"{fdel=}",
            f"{doc=}",
            sep="\n\t",
        )
        self.fget = fget
        self.fset = fset
        self.fdel = fdel
        if doc is None and fget is not None:
            doc = fget.__doc__
        self.__doc__ = doc
        # self.decorated_obj = None

    #     def __call__(self, *args, **kwargs):
    #         return self.fget.__call__(*args, **kwargs)

    def __get__(self, obj, objtype=None):
        print("Property.__get__:", f"{self=}", f"{obj=}", f"{objtype=}", sep="\n\t")
        if self.fget is None:
            raise AttributeError("unreadable attribute")
        if obj is None:
            result = self
        else:
            # self.decorated_obj = obj
            result = self.fget(obj)
        print(f">>> Property.__get__ ", f"{(obj is None)=}", f"{result=}", sep="\n\t")
        return result

    def __set__(self, obj, value):
        if self.fset is None:
            raise AttributeError("can't set attribute")
        self.fset(obj, value)

    def __delete__(self, obj):
        if self.fdel is None:
            raise AttributeError("can't delete attribute")
        self.fdel(obj)

    def getter(self, fget):
        return type(self)(fget, self.fset, self.fdel, self.__doc__)

    def setter(self, fset):
        return type(self)(self.fget, fset, self.fdel, self.__doc__)

    def deleter(self, fdel):
        return type(self)(self.fget, self.fset, fdel, self.__doc__)

In [20]:
class ClassMethod(property):
    "Emulate PyClassMethod_Type() in Objects/funcobject.c"

    def __init__(self, fget=None, fset=None, fdel=None, doc=None):
        print(
            "ClassMethod.__init__:",
            f"{self=}",
            f"{fget=}",
            f"{fset=}",
            f"{fdel=}",
            f"{doc=}",
            sep="\n\t",
        )
        super().__init__(fget, fset, fdel, doc)

        if doc is None and fget is not None:
            doc = fget.__doc__
        self.__doc__ = doc

    # def __call__(self, *args, **kwargs):
    #     return self.f.__call__(*args, **kwargs)

    def __get__(self, obj, cls=None):
        print(f"ClassMethod.__get__:", f"{self=}", f"{obj=}", f"{cls=}", sep="\n\t")
        if cls is None:
            cls = type(obj)
        if hasattr(type(self.fget), "__get__"):
            # result = self.f.__get__(cls)
            print("BRANCH 1")
            result = MethodType(self.fget.__get__, cls).__call__()
        else:
            print("BRANCH 2")
            result = MethodType(self.fget, cls)
        print(
            f">>> ClassMethod.__get__:",
            f"{hasattr(type(self.fget), '__get__')=}",
            f"{result=}",
            sep="\n\t",
        )
        return result

In [21]:
class LazyAttribute(property):
    def __init__(self, fget=None, fset=None, fdel=None, doc=None):
        print(
            "ClassMethod.__init__:",
            f"{self=}",
            f"{fget=}",
            f"{fset=}",
            f"{fdel=}",
            f"{doc=}",
            sep="\n\t",
        )

        super().__init__(fget, fset, fdel, doc)

        if doc is None and fget is not None:
            doc = fget.__doc__
        self.__doc__ = doc

    def __get__(self, obj, cls=None):
        print(f"ClassMethod.__get__:", f"{self=}", f"{obj=}", f"{cls=}", sep="\n\t")
        if cls is None:
            cls = type(obj)

        result = MethodType(property(self.fget).__get__, cls).__call__()
        # if hasattr(type(self.fget), "__get__"):
        #     # result = self.f.__get__(cls
        #     result = MethodType(self.fget.__get__, cls).__call__()
        # else:
        #     result = MethodType(self.fget, cls)
        print(
            f">>> ClassMethod.__get__:",
            f"{hasattr(type(self.fget), '__get__')=}",
            f"{result=}",
            sep="\n\t",
        )
        return result

In [22]:
class ClassProperty(property):
    def __init__(self, fget=None, fset=None, fdel=None, doc=None):
        fget = property(fget)
        super().__init__(fget, fset, fdel, doc)
        self.__doc__ = fget.__doc__ if doc is None else doc

    def __get__(self, obj, cls=None):
        if cls is None:
            cls = type(obj)
        if self.fget is None:
            raise AttributeError("unreadable attribute")
        return MethodType(self.fget.__get__, cls).__call__()

    def __set__(self, obj, value):
        print(f"__SET__ called", f"{obj=}", f"{value=}", f"{self.fset=}", sep="\t\n")
        if self.fset is None:
            raise AttributeError("can't set attribute")
        self.fset(obj, value)

    def __delete__(self, obj):
        if self.fdel is None:
            raise AttributeError("can't delete attribute")
        self.fdel(obj)

    def getter(self, fget):
        return type(self)(fget, self.fset, self.fdel, self.__doc__)

    def setter(self, fset):
        return type(self)(self.fget, fset, self.fdel, self.__doc__)

    def deleter(self, fdel):
        return type(self)(self.fget, self.fset, fdel, self.__doc__)

In [23]:
class Testit:
    @ClassProperty
    def myclsprop(cls):
        """My Class-Property"""
        return f"My name is {cls.__name__}"


display(Testit.__dict__)
display(Testit.__dict__["myclsprop"].__dir__())
display(Testit.myclsprop)

mappingproxy({'__module__': '__main__',
              'myclsprop': <__main__.ClassProperty at 0x7fe80f6fdee0>,
              '__dict__': <attribute '__dict__' of 'Testit' objects>,
              '__weakref__': <attribute '__weakref__' of 'Testit' objects>,
              '__doc__': None})

['__doc__',
 '__module__',
 '__init__',
 '__get__',
 '__set__',
 '__delete__',
 'getter',
 'setter',
 'deleter',
 '__dict__',
 '__weakref__',
 '__getattribute__',
 '__new__',
 'fget',
 'fset',
 'fdel',
 '__isabstractmethod__',
 '__repr__',
 '__hash__',
 '__str__',
 '__setattr__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__eq__',
 '__ne__',
 '__gt__',
 '__ge__',
 '__reduce_ex__',
 '__reduce__',
 '__subclasshook__',
 '__init_subclass__',
 '__format__',
 '__sizeof__',
 '__dir__',
 '__class__']

'My name is Testit'

In [24]:
Testit.__dict__["myclsprop"].fset is None

True

In [25]:
help(Testit)

Help on class Testit in module __main__:

class Testit(builtins.object)
 |  Readonly properties defined here:
 |  
 |  myclsprop
 |      My Class-Property
 |  
 |  ----------------------------------------------------------------------
 |  Data descriptors defined here:
 |  
 |  __dict__
 |      dictionary for instance variables (if defined)
 |  
 |  __weakref__
 |      list of weak references to the object (if defined)



In [26]:
Testit.myclsprop = 2

In [27]:
Testit.myclsprop = 2

In [28]:
Testit.__dict__

mappingproxy({'__module__': '__main__',
              'myclsprop': 2,
              '__dict__': <attribute '__dict__' of 'Testit' objects>,
              '__weakref__': <attribute '__weakref__' of 'Testit' objects>,
              '__doc__': None})

In [29]:
class ClassMethod(property):
    "Emulate PyClassMethod_Type() in Objects/funcobject.c"

    def __init__(self, func):
        self.func = classmethod(func)

    def __get__(self, obj, cls=None):
        if cls is None:
            cls = type(obj)
        if hasattr(type(self.func), "__get__"):
            print("Branch 1")
            return self.func.__get__(cls)
        return MethodType(self.func, cls)

In [30]:
class Testit:
    @ClassMethod
    @property
    def myclsprop(cls):
        """My Class-Property"""
        return f"My name is {cls.__name__}"


display(Testit.__dict__)
Testit.myclsprop

mappingproxy({'__module__': '__main__',
              'myclsprop': <__main__.ClassMethod at 0x7fe80f690fa0>,
              '__dict__': <attribute '__dict__' of 'Testit' objects>,
              '__weakref__': <attribute '__weakref__' of 'Testit' objects>,
              '__doc__': None})

Branch 1


'My name is type'

In [31]:
help(Testit)

Branch 1
Branch 1
Branch 1
Help on class Testit in module __main__:

class Testit(builtins.object)
 |  Readonly properties defined here:
 |  
 |  myclsprop
 |  
 |  ----------------------------------------------------------------------
 |  Data descriptors defined here:
 |  
 |  __dict__
 |      dictionary for instance variables (if defined)
 |  
 |  __weakref__
 |      list of weak references to the object (if defined)



In [34]:
Testit.__dict__["myclsprop"]

In [35]:
from time import sleep
from abc import ABC, ABCMeta, abstractmethod


class MyMetaClass(ABCMeta):
    @classmethod
    @property
    def expensive_metaclass_property(cls):
        """This may take a while to compute!"""
        print("computing metaclass property")
        sleep(3)
        return "Phew, that was a lot of work!"


class MyBaseClass(ABC, metaclass=MyMetaClass):
    @classmethod
    @property
    def expensive_class_property(cls):
        """This may take a while to compute!"""
        print("computing class property ..")
        sleep(3)
        return "Phew, that was a lot of work!"

    @property
    def expensive_instance_property(self):
        """This may take a while to compute!"""
        print("computing instance property ...")
        sleep(3)
        return "Phew, that was a lot of work!"


class MyClass(MyBaseClass):
    """Some subclass of MyBaseClass"""

In [36]:
help(MyClass)

computing class property ..
computing class property ..
computing class property ..
computing class property ..
computing class property ..
Help on class MyClass in module __main__:

class MyClass(MyBaseClass)
 |  Some subclass of MyBaseClass
 |  
 |  Method resolution order:
 |      MyClass
 |      MyBaseClass
 |      abc.ABC
 |      builtins.object
 |  
 |  Data and other attributes defined here:
 |  
 |  __abstractmethods__ = frozenset()
 |  
 |  ----------------------------------------------------------------------
 |  Class methods inherited from MyBaseClass:
 |  
 |  expensive_class_property = 'Phew, that was a lot of work!'
 |  ----------------------------------------------------------------------
 |  Readonly properties inherited from MyBaseClass:
 |  
 |  expensive_instance_property
 |      This may take a while to compute!
 |  
 |  ----------------------------------------------------------------------
 |  Data descriptors inherited from MyBaseClass:
 |  
 |  __dict__
 |      

In [37]:
MyBaseClass.expensive_class_property = 2
MyBaseClass().expensive_instance_property = 2
# MyBaseClass.__dict__

AttributeError: can't set attribute

In [38]:
instance = MyBaseClass()
instance.expensive_class_property = 2
instance.expensive_instance_property = 2

AttributeError: can't set attribute

In [39]:
import math


class TrigConst:
    def __init__(self, const=math.pi):
        self.const = const

    def __get__(self, obj, objtype=None):
        return self.const

    def __set__(self, obj, value):
        print(f"__set__ called", f"{obj=}")
        self.const = value


class Trig:
    const = TrigConst()


class PatchedSetattr(type):
    def __setattr__(cls, key, value):
        if hasattr(cls, key):
            obj = cls.__dict__[key]
            if hasattr(obj, "__set__"):
                obj.__set__(cls, value)
        else:
            super().__setattr__(key, value)


class PatchedTrig(metaclass=PatchedSetattr):
    const = TrigConst()

In [40]:
inst = Trig()
print(inst.const, type(inst.__class__.__dict__["const"]))
inst.const = math.tau
print(inst.const, type(inst.__class__.__dict__["const"]))

3.141592653589793 <class '__main__.TrigConst'>
__set__ called obj=<__main__.Trig object at 0x7fe80f683280>
6.283185307179586 <class '__main__.TrigConst'>


In [41]:
cls = Trig
print(cls.const, type(cls.__dict__["const"]))
cls.const = math.tau
print(cls.const, type(cls.__dict__["const"]))

6.283185307179586 <class '__main__.TrigConst'>
6.283185307179586 <class 'float'>


In [42]:
inst = PatchedTrig()
print(inst.const, type(inst.__class__.__dict__["const"]))
inst.const = math.tau
print(inst.const, type(inst.__class__.__dict__["const"]))

3.141592653589793 <class '__main__.TrigConst'>
__set__ called obj=<__main__.PatchedTrig object at 0x7fe80f683eb0>
6.283185307179586 <class '__main__.TrigConst'>


In [43]:
cls = PatchedTrig
print(cls.const, type(cls.__dict__["const"]))
cls.const = math.tau
print(cls.const, type(cls.__dict__["const"]))

6.283185307179586 <class '__main__.TrigConst'>
__set__ called obj=<class '__main__.PatchedTrig'>
6.283185307179586 <class '__main__.TrigConst'>


In [44]:
import logging

logging.basicConfig(level=logging.INFO)


class LoggedAgeAccess:
    def __get__(self, obj, objtype=None):
        value = obj._age
        logging.info("Accessing %r giving %r", "age", value)
        return value

    def __set__(self, obj, value):
        logging.info("Updating %r to %r", "age", value)
        obj._age = value


class Person:

    age = LoggedAgeAccess()  # Descriptor instance

    def __init__(self, name, age):
        self.name = name  # Regular instance attribute
        self.age = age  # Calls __set__()

    def birthday(self):
        self.age += 1  # Calls both __get__() and __set__()

In [46]:
import logging

logging.basicConfig(level=logging.INFO)


class LoggedAgeAccess:
    def __get__(self, obj, objtype=None):
        value = obj._age
        logging.info("Accessing %r giving %r", "age", value)
        return value

    def __set__(self, obj, value):
        logging.info("Updating %r to %r", "age", value)
        obj._age = value


class Person:

    age = LoggedAgeAccess()  # Descriptor instance

    def __init__(self, name, age):
        self.name = name  # Regular instance attribute
        self.age = age  # Calls __set__()

    def birthday(self):
        self.age += 1  # Calls both __get__() and __set__()

In [47]:
import logging

logging.basicConfig(level=logging.INFO)


class LoggedAgeAccess:
    def __get__(self, obj, objtype=None):
        value = obj._age
        logging.info("Accessing %r giving %r", "age", value)
        return value

    def __set__(self, obj, value):
        logging.info("Updating %r to %r", "age", value)
        obj._age = value


class Person:

    age = LoggedAgeAccess()  # Descriptor instance

    def __init__(self, name, age):
        self.name = name  # Regular instance attribute
        self.age = age  # Calls __set__()

    def birthday(self):
        self.age += 1  # Calls both __get__() and __set__()